##ARAGEN LIFE SCIENCES

TASK 2 - ETL PIPELINE

Extract -> Transform -> Load

Input : lab_data.csv

Output : lab_quality.db

Author : Miya

###Import Libraries

In [1]:
import pandas as pd
import numpy as np
import sqlite3

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


###Upload Dataset

In [3]:
from google.colab import files
uploaded = files.upload()

Saving lab_data.csv to lab_data.csv


###Extraction Phase

In [4]:
df = pd.read_csv("lab_data.csv")
print("Dataset Loaded Successfully")
print("Shape:", df.shape)
df.head()

Dataset Loaded Successfully
Shape: (500000, 10)


,sample_id,experiment_date,instrument_id,lab_name,analyte_name,measured_value,unit,operator_name,status,recorded_at
0,LAB-000001,2026-05-24,INST-LC-010,Formulation Lab,Ibuprofen,107.45,g/L,Jason Poole,Pass,2026-05-24 07:00:00
1,LAB-000002,2025-08-27,INST-LC-001,Analytical Lab A,Doxycycline,97.93,ug/mL,Chad Williams,Pass,2025-08-27 08:00:00
2,LAB-000003,2025-07-13,INST-LC-005,Stability Lab,Diclofenac,109.72,ppm,Jason Poole,Pass,2025-07-13 12:00:00
3,LAB-000004,2025-11-18,INST-LC-003,Analytical Lab A,Diclofenac,122.85,ppm,Jennifer Olson,Fail,2025-11-18 10:00:00
4,LAB-000005,2025-11-03,INST-LC-008,Analytical Lab A,Doxycycline,96.49,ppm,Jennifer Jensen,Pending,2025-11-03 01:00:00


###Data Inspection

In [5]:
print("\nColumn Names")
print(df.columns)

print("\nData Types")
print(df.dtypes)

print("\nNull Values")
print(df.isnull().sum())


Column Names
Index(['sample_id', 'experiment_date', 'instrument_id', 'lab_name',
       'analyte_name', 'measured_value', 'unit', 'operator_name', 'status',
       'recorded_at'],
      dtype='object')

Data Types
sample_id           object
experiment_date     object
instrument_id       object
lab_name            object
analyte_name        object
measured_value     float64
unit                object
operator_name       object
status              object
recorded_at         object
dtype: object

Null Values
sample_id              0
experiment_date        0
instrument_id      25000
lab_name               0
analyte_name           0
measured_value     24507
unit               25000
operator_name      25000
status                 0
recorded_at            0
dtype: int64


###Transformation Phase (Data Cleaning)

In [6]:
df["experiment_date"] = pd.to_datetime(
    df["experiment_date"],
    errors="coerce"
)

df["recorded_at"] = pd.to_datetime(
    df["recorded_at"],
    errors="coerce"
)

print("Dates Standardized")

Dates Standardized


###Clean Lab Name

In [7]:
df["lab_name"] = (
    df["lab_name"]
    .astype(str)
    .str.strip()
    .str.title()
)

print("Lab Names Standardized")

Lab Names Standardized


###Clean Status Column

In [8]:
df["status"] = (
    df["status"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(df["status"].value_counts())

status
Pass       374940
Fail        75141
Pending     49919
Name: count, dtype: int64


###Remove Extra Space

In [9]:
string_columns = [
    "sample_id",
    "instrument_id",
    "lab_name",
    "analyte_name",
    "unit",
    "operator_name",
    "status"
]

for col in string_columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

print("Whitespace Removed")

Whitespace Removed


###Validation Of Transformed Data

In [10]:
print("Dataset Shape")
print(df.shape)

print("\nData Types")
print(df.dtypes)

print("\nNull Values")
print(df.isnull().sum())

Dataset Shape
(500000, 10)

Data Types
sample_id                  object
experiment_date    datetime64[ns]
instrument_id              object
lab_name                   object
analyte_name               object
measured_value            float64
unit                       object
operator_name              object
status                     object
recorded_at        datetime64[ns]
dtype: object

Null Values
sample_id              0
experiment_date        0
instrument_id          0
lab_name               0
analyte_name           0
measured_value     24507
unit                   0
operator_name          0
status                 0
recorded_at            0
dtype: int64


###Dimension Table Creation

####Dimension 1

In [11]:
dim_lab = (
    df[["lab_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_lab["lab_key"] = range(
    1,
    len(dim_lab)+1
)

dim_lab = dim_lab[
    ["lab_key","lab_name"]
]

dim_lab.head()

,lab_key,lab_name
0,1,Formulation Lab
1,2,Analytical Lab A
2,3,Stability Lab
3,4,Microbiology Lab
4,5,Analytical Lab B


####Dimension 2

In [12]:
dim_instrument = (
    df[["instrument_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_instrument["instrument_key"] = range(
    1,
    len(dim_instrument)+1
)

dim_instrument = dim_instrument[
    ["instrument_key","instrument_id"]
]

dim_instrument.head()

,instrument_key,instrument_id
0,1,INST-LC-010
1,2,INST-LC-001
2,3,INST-LC-005
3,4,INST-LC-003
4,5,INST-LC-008


####Dimension 3

In [13]:
dim_operator = (
    df[["operator_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_operator["operator_key"] = range(
    1,
    len(dim_operator)+1
)

dim_operator = dim_operator[
    ["operator_key","operator_name"]
]

dim_operator.head()

,operator_key,operator_name
0,1,Jason Poole
1,2,Chad Williams
2,3,Jennifer Olson
3,4,Jennifer Jensen
4,5,Nancy Schwartz


###Fact Table Creation

In [14]:
fact_df = df.merge(
    dim_lab,
    on="lab_name",
    how="left"
)

fact_df = fact_df.merge(
    dim_instrument,
    on="instrument_id",
    how="left"
)

fact_df = fact_df.merge(
    dim_operator,
    on="operator_name",
    how="left"
)

fact_df.head()

,sample_id,experiment_date,instrument_id,lab_name,analyte_name,measured_value,unit,operator_name,status,recorded_at,lab_key,instrument_key,operator_key
0,LAB-000001,2026-05-24,INST-LC-010,Formulation Lab,Ibuprofen,107.45,g/L,Jason Poole,Pass,2026-05-24 07:00:00,1,1,1
1,LAB-000002,2025-08-27,INST-LC-001,Analytical Lab A,Doxycycline,97.93,ug/mL,Chad Williams,Pass,2025-08-27 08:00:00,2,2,2
2,LAB-000003,2025-07-13,INST-LC-005,Stability Lab,Diclofenac,109.72,ppm,Jason Poole,Pass,2025-07-13 12:00:00,3,3,1
3,LAB-000004,2025-11-18,INST-LC-003,Analytical Lab A,Diclofenac,122.85,ppm,Jennifer Olson,Fail,2025-11-18 10:00:00,2,4,3
4,LAB-000005,2025-11-03,INST-LC-008,Analytical Lab A,Doxycycline,96.49,ppm,Jennifer Jensen,Pending,2025-11-03 01:00:00,2,5,4


###Fact Table Final Structure

In [15]:
fact_lab_measurements = fact_df[[
    "sample_id",
    "lab_key",
    "instrument_key",
    "operator_key",
    "experiment_date",
    "recorded_at",
    "analyte_name",
    "measured_value",
    "unit",
    "status"
]]

fact_lab_measurements.head()

,sample_id,lab_key,instrument_key,operator_key,experiment_date,recorded_at,analyte_name,measured_value,unit,status
0,LAB-000001,1,1,1,2026-05-24,2026-05-24 07:00:00,Ibuprofen,107.45,g/L,Pass
1,LAB-000002,2,2,2,2025-08-27,2025-08-27 08:00:00,Doxycycline,97.93,ug/mL,Pass
2,LAB-000003,3,3,1,2025-07-13,2025-07-13 12:00:00,Diclofenac,109.72,ppm,Pass
3,LAB-000004,2,4,3,2025-11-18,2025-11-18 10:00:00,Diclofenac,122.85,ppm,Fail
4,LAB-000005,2,5,4,2025-11-03,2025-11-03 01:00:00,Doxycycline,96.49,ppm,Pending


###Star Schema

In [16]:
print("dim_lab")
print(dim_lab.shape)

print("\ndim_instrument")
print(dim_instrument.shape)

print("\ndim_operator")
print(dim_operator.shape)

print("\nfact_lab_measurements")
print(fact_lab_measurements.shape)

dim_lab
(7, 2)

dim_instrument
(16, 2)

dim_operator
(26, 2)

fact_lab_measurements
(500000, 10)


###Load Phase(SQLITE Database)

In [17]:
conn = sqlite3.connect(
    "lab_quality.db"
)

print("Database Created")

Database Created


###Load Tables into the Database

In [18]:
dim_lab.to_sql(
    "dim_lab",
    conn,
    if_exists="replace",
    index=False
)

dim_instrument.to_sql(
    "dim_instrument",
    conn,
    if_exists="replace",
    index=False
)

dim_operator.to_sql(
    "dim_operator",
    conn,
    if_exists="replace",
    index=False
)

fact_lab_measurements.to_sql(
    "fact_lab_measurements",
    conn,
    if_exists="replace",
    index=False
)

print("All Tables Loaded Successfully")

All Tables Loaded Successfully


###Database Verification

In [19]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql(
    query,
    conn
)

tables

,name
0,dim_lab
1,dim_instrument
2,dim_operator
3,fact_lab_measurements


###Save Database

In [20]:
conn.close()
print("Database Saved")

Database Saved


###Download Database

In [21]:
from google.colab import files
files.download(
    "lab_quality.db"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>